# Path A, P10 — Decision 1: R0 vs R1 (Oct 27, 2026)

Pre-registered in the Path A Workbook, Ch. A8.2. No choice below was changed.

- **Data:** `data/processed/hazard_table.parquet` (P8), topology Hierarchy or Full, all rows (bot runs and incomplete runs kept, as in the table). G-train rows fit the models; G-val rows score them. Outcome `y_rev`. lab = 1 if dataset is EL or CL, 0 if CC.
- **Features** (raw, no scaling): A, trend (w = 1 s), |s| (`abs_s`), `at_end`, e = log(1 + time at end) (`time_at_end_log1p`), r = log(1 + time since last reversal or first move) (`time_since_rev_log1p`).
- **Models** (unpenalized logistic, statsmodels Logit): R0 = const + lab + 6 features (8 params); R1 = R0 + lab x each feature (14 params). R1 cloud weights = beta_j; lab weights = beta_j + gamma_j.
- **delta** = sum over G-val bins of (ln L_R1 − ln L_R0) / n bins (per-bin log-likelihood y ln h + (1 − y) ln(1 − h)). 95% CI: 10,000 resamples of G-val groups (log files) with replacement, seed 573; betas fixed, no refitting.
- **Verdict rule:** CI includes 0 or delta <= 0 → (b); CI entirely > 0 → (a).
- **Runs:** primary (trend, w = 1 s); sensitivity (`trend_scaled` in place of trend).

Functions: `fit.fit`, `fit.bin_ll`, `fit.boot_delta` (`src/slider/fit.py`). Saved: `results/tables/p10_decision1.csv`, `results/tables/p10_coefs.csv`.

In [1]:
import warnings

import numpy as np
import pandas as pd
from statsmodels.tools.sm_exceptions import PerfectSeparationWarning

from slider import config
from slider import fit

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)

ht = pd.read_parquet(config.HAZARD_TABLE)
ht = ht[ht["topology"].isin(["Hierarchy", "Full"])]
train, val = ht[ht["split"] == "train"], ht[ht["split"] == "val"]
lab_train = train["dataset"].isin(["EL", "CL"])

ht.groupby(["split", "topology", "dataset"]).agg(
    rows=("y_rev", "size"), sum_y_rev=("y_rev", "sum"), groups=("group_id", "nunique"),
    rows_bot_run=("bot_run", "sum"), rows_incomplete=("incomplete", "sum"))

rows  sum_y_rev  groups  rows_bot_run  rows_incomplete
split topology  dataset                                                         
train Full      CC       59752       1395      17         13360             2731
                CL       80239       1477      11             0                0
      Hierarchy CC       59157        860      17         16565                0
                CL       55732        913      11             0                0
                EL       45365        848      11             0                0
val   Full      CC       19479        517       8         12819                0
                CL       77419       1372       7             0                0
      Hierarchy CC       25336        550       8         12495                0
                CL       31688        311       7             0                0
                EL       50969        756       6             0                0

## Fits on G-train, delta on G-val

In [2]:
P8_TRAIN_ROWS, P8_TRAIN_Y_REV = 300245, 5493    # G-train, Hierarchy + Full: p8_hazard_table.ipynb, rows and sum_y_rev by split


def fit_logged(df, feats, lab_interact):
    """fit.fit, keeping every warning it raises."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model = fit.fit(df, feats, lab_interact)
    return model, caught


def run(name, trend):
    feats = ["A", trend, "abs_s", "at_end", "time_at_end_log1p", "time_since_rev_log1p"]
    (r0, w0), (r1, w1) = fit_logged(train, feats, False), fit_logged(train, feats, True)
    checks = []

    def check(number, what, numbers, ok):
        checks.append({"run": name, "check": number, "what": what, "numbers": numbers, "pass": bool(ok)})

    # 1. converged, finite betas, no separation warnings
    for label, m, w in [("R0", r0, w0), ("R1", r1, w1)]:
        sep = [x for x in w if issubclass(x.category, PerfectSeparationWarning)]
        check(1, f"{label}: converged, betas finite, no separation warning",
              f"converged={m.mle_retvals['converged']}, iterations={m.mle_retvals['iterations']}, "
              f"finite={np.isfinite(m.params).all()}, separation warnings={len(sep)}, all warnings={len(w)}"
              + "".join(f" | {x.category.__name__}: {x.message}" for x in w),
              m.mle_retvals["converged"] and np.isfinite(m.params).all() and not sep)
    # 2. calibration in the large on G-train: overall, lab, cloud
    y = train["y_rev"].to_numpy()
    for label, m in [("R0", r0), ("R1", r1)]:
        h = np.asarray(m.predict())
        for part, mask in [("overall", np.ones(len(train), bool)), ("lab", lab_train.to_numpy()), ("cloud", ~lab_train.to_numpy())]:
            check(2, f"{label}: sum(h_hat) == sum(y_rev), {part}",
                  f"sum h = {h[mask].sum():.6f}, sum y = {y[mask].sum()}, diff = {h[mask].sum() - y[mask].sum():.2e}",
                  abs(h[mask].sum() - y[mask].sum()) < 1e-3)
    # 3. G-train LL: R1 >= R0
    check(3, "G-train LL: R1 >= R0", f"LL R0 = {r0.llf:.4f}, LL R1 = {r1.llf:.4f}, R1 - R0 = {r1.llf - r0.llf:.4f}",
          r1.llf >= r0.llf)
    # 4. signs and size of the R0 weights
    bA, bt = r0.params["A"], r0.params[trend]
    check(4, "R0: beta_A < 0", f"beta_A = {bA:.4f}", bA < 0)
    check(4, "R0: beta_A roughly -1 to -4", f"beta_A = {bA:.4f}", -4 <= bA <= -1)
    check(4, f"R0: beta_{trend} < 0", f"beta_{trend} = {bt:.4f}", bt < 0)
    # 5. G-train rows and sum(y_rev) in the fit = the hazard table subset (and P8)
    for label, m in [("R0", r0), ("R1", r1)]:
        n, s = int(m.nobs), int(m.model.endog.sum())
        check(5, f"{label}: G-train rows and sum(y_rev) = hazard table subset = P8",
              f"fit: {n} rows, {s} reversals | table: {len(train)}, {int(train['y_rev'].sum())} | P8: {P8_TRAIN_ROWS}, {P8_TRAIN_Y_REV}",
              n == len(train) == P8_TRAIN_ROWS and s == train["y_rev"].sum() == P8_TRAIN_Y_REV)

    # delta (R1 - R0) on G-val, CI by resampling G-val groups
    delta, (low, high) = fit.boot_delta(fit.bin_ll(r1, val), fit.bin_ll(r0, val), val["group_id"])
    decision = {"run": name, "trend_feature": trend, "delta": delta, "ci_low": low, "ci_high": high,
                "n_val_bins": len(val), "n_val_groups": val["group_id"].nunique(),
                "verdict": "(a)" if low > 0 and delta > 0 else "(b)"}

    # weights: R0 with SEs; R1 cloud (beta_j) and lab (beta_j + gamma_j)
    terms = ["const", "lab"] + feats
    coefs = pd.DataFrame({"run": name, "term": terms, "R0": r0.params[terms].to_numpy(), "R0_se": r0.bse[terms].to_numpy(),
                          "R1_cloud": [r1.params["const"], np.nan] + [r1.params[f] for f in feats],
                          "R1_lab": [r1.params["const"] + r1.params["lab"], np.nan]
                                    + [r1.params[f] + r1.params["lab:" + f] for f in feats]})
    return pd.DataFrame(checks), decision, coefs


results = [run("primary", "trend"), run("sensitivity", "trend_scaled")]
checks = pd.concat([r[0] for r in results], ignore_index=True)
decision = pd.DataFrame([r[1] for r in results])
coefs = pd.concat([r[2] for r in results], ignore_index=True)

## Checks

In [3]:
with pd.option_context("display.max_colwidth", 200):
    display(checks)
print("all checks pass:", bool(checks["pass"].all()))
checks.groupby(["run", "check"])["pass"].all().unstack("run")

,run,check,what,numbers,pass
0,primary,1,"R0: converged, betas finite, no separation warning","converged=True, iterations=9, finite=True, separation warnings=0, all warnings=0",True
1,primary,1,"R1: converged, betas finite, no separation warning","converged=True, iterations=9, finite=True, separation warnings=0, all warnings=0",True
2,primary,2,"R0: sum(h_hat) == sum(y_rev), overall","sum h = 5493.000000, sum y = 5493, diff = 9.09e-13",True
3,primary,2,"R0: sum(h_hat) == sum(y_rev), lab","sum h = 3238.000000, sum y = 3238, diff = 0.00e+00",True
4,primary,2,"R0: sum(h_hat) == sum(y_rev), cloud","sum h = 2255.000000, sum y = 2255, diff = 0.00e+00",True
5,primary,2,"R1: sum(h_hat) == sum(y_rev), overall","sum h = 5493.000000, sum y = 5493, diff = 0.00e+00",True
6,primary,2,"R1: sum(h_hat) == sum(y_rev), lab","sum h = 3238.000000, sum y = 3238, diff = 4.55e-13",True
7,primary,2,"R1: sum(h_hat) == sum(y_rev), cloud","sum h = 2255.000000, sum y = 2255, diff = 4.55e-13",True
8,primary,3,G-train LL: R1 >= R0,"LL R0 = -24865.1755, LL R1 = -24821.9658, R1 - R0 = 43.2097",True
9,primary,4,R0: beta_A < 0,beta_A = -1.6046,True


all checks pass: True


run,primary,sensitivity
check,,
1,True,True
2,True,True
3,True,True
4,True,True
5,True,True


## Decision 1: delta (R1 − R0) on G-val, per bin

In [4]:
decision.to_csv(config.P10_DECISION1, index=False)
with pd.option_context("display.float_format", "{:.3e}".format):
    display(decision)

,run,trend_feature,delta,ci_low,ci_high,n_val_bins,n_val_groups,verdict
0,primary,trend,1.570e-04,-1.791e-05,2.965e-04,204891,21,(b)
1,sensitivity,trend_scaled,-4.670e-06,-3.120e-04,2.102e-04,204891,21,(b)


## Weights: R0 (with SEs); R1 cloud (beta_j) and lab (beta_j + gamma_j)

For R1, the `const` row gives the cloud intercept (beta_0) and the lab intercept (beta_0 + beta_lab); the `lab` row is empty.

In [5]:
coefs.to_csv(config.P10_COEFS, index=False)
coefs.round(4)

,run,term,R0,R0_se,R1_cloud,R1_lab
0,primary,const,-3.2989,0.0587,-3.2681,-3.2978
1,primary,lab,0.0727,0.0292,NaN,NaN
2,primary,A,-1.6046,0.0537,-1.8073,-1.4606
3,primary,trend,-1.9520,0.0875,-1.4473,-2.7144
4,primary,abs_s,0.9512,0.0925,0.9508,0.9657
5,primary,at_end,0.5267,0.0643,0.3597,0.6625
6,primary,time_at_end_log1p,-0.4395,0.0256,-0.5279,-0.4557
7,primary,time_since_rev_log1p,-0.2210,0.0261,-0.0478,-0.2649
8,sensitivity,const,-3.2739,0.0588,-3.2217,-3.3297
9,sensitivity,lab,0.0204,0.0291,NaN,NaN
